In [1]:
from ultralytics import YOLO
import yaml
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
import os, random, json
from PIL import Image, ImageOps
from collections import Counter
from scipy.stats import mode
from skimage import measure

In [2]:
TRAINED_MODEL_PATH = '/workspace/user3/runs/segment/yolo26s_custom_seg_model2/weights/best.pt'

# 베스트 모델 불러오기.
loaded_model = YOLO(TRAINED_MODEL_PATH)

In [5]:
fix_label_load = pd.read_csv('/workspace/데이터셋/[노션] 3차 프로젝트(4조)-오분류 감정 라벨링.csv')
fix_label = fix_label_load.iloc[:,:5]
change_label = fix_label[~(fix_label['이전 감정']==fix_label['정정 감정'])]
box_error_img = fix_label[fix_label['BOX 이상']=='Yes']

In [6]:
ADRESS = '/workspace/데이터셋'

img_adress = f'{ADRESS}/img'
label_adress = f'{ADRESS}/label'
segmentation_adress = f'{ADRESS}/segmentation'

In [7]:
DATA_TYPES_TO_PROCESS = ['train', 'val']
EMOTIONS = ['anger', 'happy', 'sadness', 'panic']

In [8]:
def load_emotion_data_files(emotion, data_type):
    """
    주어진 감정과 데이터 유형에 대한 json 라벨과 npz 세그먼트를 불러오는 함수.
    (json_data, npz_data)가 디폴트 반환값이지만 데이터를 찾을 수 없는 경우 (None, None)을 반환
    """
    json_file_path = os.path.join(label_adress, data_type, f'{data_type}_{emotion}.json')
    npz_file_path = os.path.join(segmentation_adress, data_type, f'{data_type}_{emotion}.npz')

    current_json_data = None
    current_npz_data = None

    # json 데이터 로드
    if os.path.exists(json_file_path):
        try:
            with open(json_file_path, 'r', encoding='cp949') as f:
                current_json_data = json.load(f)
            print(f"Loaded JSON for {emotion} ({data_type}) from {json_file_path}")
        except Exception as e:
            print(f"Error loading JSON for {emotion} ({data_type}) from {json_file_path}: {e}")
    else:
        print(f"JSON file not found: {json_file_path}")

    # npz 데이터 로드
    if os.path.exists(npz_file_path):
        try:
            current_npz_data = np.load(npz_file_path)
            print(f"Loaded NPZ for {emotion} ({data_type}) from {npz_file_path}")
        except Exception as e:
            print(f"Error loading NPZ for {emotion} ({data_type}) from {npz_file_path}: {e}")
    else:
        print(f"NPZ file not found: {npz_file_path}")

    return current_json_data, current_npz_data

In [9]:
def combined_data_iterator():
    """
    이미지, 라벨, 세그먼트 데이터를 매칭시켜주는 함수
    EMOTIONS, DATA_TYPES_TO_PROCESS 변수로 지정된 데이터 유형과 감정에 대해서만 조회.
    """
    for data_type in DATA_TYPES_TO_PROCESS:
        for emotion in EMOTIONS:
            json_labels, npz_segments = load_emotion_data_files(emotion, data_type)

            # 라벨, 세그먼트 중에 하나라도 누락되어있을 경우 스킵.
            if json_labels is None or npz_segments is None:
                print(f"Skipping {emotion} {data_type} due to missing JSON or NPZ data.")
                if npz_segments is not None: npz_segments.close()
                continue

            # img 데이터 로드
            img_emotion_dir = os.path.join(img_adress, data_type, emotion)

            # 이미지가 누락되어있을 경우 스킵.
            if not os.path.exists(img_emotion_dir):
                print(f"Image directory not found: {img_emotion_dir}. Skipping this emotion for {data_type}.")
                if npz_segments is not None: npz_segments.close()
                continue

            for json_entry in json_labels:
                filename = json_entry.get('filename')
                if not filename: continue # 이미지 파일 이름이 없으면 스킵.
                
                # 재분류된 이미지 라벨 정정
                current_emotion = emotion
                if change_label.파일명.str.contains(filename.split('.')[0]).any():
                    current_emotion = change_label.loc[change_label.파일명.str.contains(filename.split('.')[0]),'정정 감정'].iloc[0]

                boxes_err = False
                if box_error_img.파일명.str.contains(filename.split('.')[0]).any(): boxes_err = True

                img_full_path = os.path.join(img_emotion_dir, filename)
                segmentation_mask = npz_segments.get(filename)

                # 파일 매칭. 이미지 + 라벨 + 세그먼트
                if os.path.exists(img_full_path) and segmentation_mask is not None:
                    entry_data = {
                        'filename': filename,
                        'data_type': data_type,
                        'gender': json_entry.get('gender'),
                        'age': json_entry.get('age'),
                        'img_path': img_full_path,
                        'faceExp': current_emotion,
                        'segmentation_mask': segmentation_mask,
                      # 'isProf': json_entry.get('isProf'),
                      # 'faceExp_uploader': json_entry.get('faceExp_uploader'),
                      # 'bg_uploader': json_entry.get('bg_uploader'),
                    }

                    min_x_vals, min_y_vals, max_x_vals, max_y_vals = [], [], [], []
                    for annotator in ['A', 'B', 'C']:
                        annot_key = f'annot_{annotator}'
                        if annot_key in json_entry and isinstance(json_entry[annot_key], dict):
                            boxes = json_entry[annot_key].get('boxes')
                            if boxes and all(k in boxes for k in ['minX', 'minY', 'maxX', 'maxY']):
                                min_x_vals.append(boxes['minX'])
                                min_y_vals.append(boxes['minY'])
                                max_x_vals.append(boxes['maxX'])
                                max_y_vals.append(boxes['maxY'])

                    avg_min_x, avg_min_y, avg_max_x, avg_max_y = None, None, None, None
                    if min_x_vals:
                        avg_min_x = np.mean(min_x_vals).item()
                        avg_min_y = np.mean(min_y_vals).item()
                        avg_max_x = np.mean(max_x_vals).item()
                        avg_max_y = np.mean(max_y_vals).item()
                        entry_data['boxes'] = {'maxX': avg_min_x, 'maxY': avg_min_y, 'minX': avg_max_x, 'minY': avg_max_y}
                        entry_data['boxes_err'] = boxes_err

                    yield entry_data

            if npz_segments is not None: npz_segments.close()

In [11]:
_original_data_types_to_process = DATA_TYPES_TO_PROCESS
_original_emotions = EMOTIONS
DATA_TYPES_TO_PROCESS = ['val']
EMOTIONS = ['anger']

temp_val_iterator = combined_data_iterator()

In [13]:
temp_val_iterator = next(temp_val_iterator)

In [16]:
temp_val_iterator

{'filename': 'qokw201acdd0f74c7794a9ddcbc9cabdf9c79dd711cbed27d0cd91a20bf19z1bj.jpg',
 'data_type': 'val',
 'gender': '여',
 'age': 20,
 'img_path': '/workspace/데이터셋/img/val/anger/qokw201acdd0f74c7794a9ddcbc9cabdf9c79dd711cbed27d0cd91a20bf19z1bj.jpg',
 'faceExp': 'anger',
 'segmentation_mask': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [4, 4, 4, ..., 0, 0, 0],
        [4, 4, 4, ..., 0, 0, 0],
        [4, 4, 4, ..., 0, 0, 0]], shape=(2208, 2944), dtype=uint8),
 'boxes': {'maxX': 950.1564562490962,
  'maxY': 708.1296366303209,
  'minX': 1715.9310818999575,
  'minY': 1671.1841681382957},
 'boxes_err': False}

In [18]:
TEST_IMAGE_PATH = temp_val_iterator['img_path']

# 원래대로 복원
DATA_TYPES_TO_PROCESS = _original_data_types_to_process
EMOTIONS = _original_emotions

results = loaded_model.predict(source=TEST_IMAGE_PATH, save=True, conf=0.25, iou=0.7)
print(temp_val_iterator['faceExp'])


image 1/1 /workspace/데이터셋/img/val/anger/qokw201acdd0f74c7794a9ddcbc9cabdf9c79dd711cbed27d0cd91a20bf19z1bj.jpg: 768x1024 1 anger, 12.1ms
Speed: 6.6ms preprocess, 12.1ms inference, 0.9ms postprocess per image at shape (1, 3, 768, 1024)
Results saved to /workspace/user3/runs/segment/predict3
anger


In [21]:
for r in results:
    # 1. 탐지된 객체가 있는지 확인
    if r.boxes is not None:
        for box in r.boxes:
            # 현재 1순위 클래스 인덱스와 확률
            cls_id = int(box.cls[0])
            conf_1st = float(box.conf[0])
            label_1st = r.names[cls_id]
            print(f"Detected: {label_1st} with confidence {conf_1st:.4f}")

Detected: anger with confidence 0.7592


In [19]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'blank', 1: 'anger', 2: 'happy', 3: 'sadness', 4: 'panic'}
 obb: None
 orig_img: array([[[201, 192, 188],
         [202, 193, 189],
         [204, 195, 191],
         ...,
         [210, 202, 202],
         [211, 203, 203],
         [210, 202, 202]],
 
        [[201, 192, 188],
         [199, 190, 186],
         [202, 193, 189],
         ...,
         [210, 202, 202],
         [211, 203, 203],
         [210, 202, 202]],
 
        [[202, 193, 189],
         [200, 191, 187],
         [202, 193, 189],
         ...,
         [209, 201, 201],
         [210, 202, 202],
         [210, 202, 202]],
 
        ...,
 
        [[130, 129, 149],
         [128, 127, 147],
         [138, 137, 157],
         ...,
         [ 54,  76,  82],
         [ 51,  73,  79],
         [ 48,  70,  76]],
 
        [[123, 121, 143],


- checkpoint: 세그먼트 모델이라 확률분포를 가져올 수 없다는 걸 깨닫고...